In [1]:
from google.colab import drive
drive.mount("/content/drive")

!pip install transformers datasets accelerate -q

DATA_DIR2 = "/content/drive/MyDrive/FactLens_Group9/data2"
ARTIFACTS = DATA_DIR2 + "/artifacts"

import os
os.makedirs(ARTIFACTS, exist_ok=True)

print("Drive mounted successfully")
print(f"Data directory: {DATA_DIR2}")

Mounted at /content/drive
Drive mounted successfully
Data directory: /content/drive/MyDrive/FactLens_Group9/data2


# Left vs Right — DistilBERT fine-tuning (optional)

Requires: `pip install -r requirements.txt` and a **CUDA-enabled PyTorch** build if you want GPU training (the default `pip install torch` is often CPU-only).

**GPU setup:** Pick the command for your CUDA version from [pytorch.org/get-started/locally](https://pytorch.org/get-started/locally/), install it in this project’s `.venv`, then **restart the kernel**. Verify with `nvidia-smi` in a terminal.

In the next cell, **`PREFER_GPU = True`** makes the notebook **require** CUDA; set it to `False` only if you intentionally train on CPU.

**Run:** *Run All* after dependencies and kernel restart.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizer,
    Trainer,
    TrainingArguments,
)

CSV_PATH = f"{DATA_DIR2}/news_bias.csv"

# Auto-detect GPU
if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
    BATCH_SIZE = 16
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    BATCH_SIZE = 8
    print("No GPU found — training on CPU (will be slow)")

MAX_SAMPLES = None  # set e.g. 4000 for a quick test run
NUM_EPOCHS = 3      # increased from 2 to match fake/real model

print(f"Training device: {DEVICE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Data: {CSV_PATH}")

GPU available: NVIDIA A100-SXM4-40GB
Training device: cuda:0
Batch size: 16
Epochs: 3
Data: /content/drive/MyDrive/FactLens_Group9/data2/news_bias.csv


In [3]:
df = pd.read_csv(CSV_PATH)
df = df[df["label"].isin(["Left", "Right"])].copy()
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 50]

if MAX_SAMPLES is not None and len(df) > MAX_SAMPLES:
    df = df.sample(MAX_SAMPLES, random_state=42)

df["labels"] = df["label"].map({"Left": 0, "Right": 1})
print(f"{len(df):,} rows")
print(df["label"].value_counts())

13,366 rows
label
Left     7803
Right    5563
Name: count, dtype: int64


In [4]:
train_df, test_df = train_test_split(
    df[["text", "labels"]],
    test_size=0.2,
    random_state=42,
    stratify=df["labels"],
)
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")


def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding=True,
        truncation=True,
        max_length=256,
    )

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))
train_ds = train_ds.map(tokenize, batched=True, batch_size=256)
test_ds = test_ds.map(tokenize, batched=True, batch_size=256)
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
print("Tokenized.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/10692 [00:00<?, ? examples/s]

Map:   0%|          | 0/2674 [00:00<?, ? examples/s]

Tokenized.


In [5]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)
model = model.to(DEVICE)

use_fp16 = DEVICE.type == "cuda"

training_args = TrainingArguments(
    output_dir=f"{ARTIFACTS}/distilbert_checkpoints",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    warmup_steps=200,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=use_fp16,
    report_to="none",
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, pred),
        "f1": f1_score(labels, pred, average="macro"),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

print("Starting training...")
print(f"This will take ~10-15 minutes on A100...")
trainer.train()
print("Training finished.")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training...
This will take ~10-15 minutes on A100...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Training finished.


In [6]:
out = trainer.predict(test_ds)
y_hat = np.argmax(out.predictions, axis=-1)
y_true = test_df["labels"].values

distilbert_acc = accuracy_score(y_true, y_hat)
distilbert_f1 = f1_score(y_true, y_hat, average="macro")

print(f"Accuracy: {distilbert_acc:.4f} ({distilbert_acc:.2%})")
print(f"Macro F1: {distilbert_f1:.4f}")
print()
print(classification_report(y_true, y_hat, target_names=["Left", "Right"]))

# Save model
save_path = f"{ARTIFACTS}/distilbert_left_right"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")

Accuracy: 0.8657 (86.57%)
Macro F1: 0.8607

              precision    recall  f1-score   support

        Left       0.87      0.91      0.89      1561
       Right       0.86      0.81      0.83      1113

    accuracy                           0.87      2674
   macro avg       0.86      0.86      0.86      2674
weighted avg       0.87      0.87      0.87      2674



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/drive/MyDrive/FactLens_Group9/data2/artifacts/distilbert_left_right


In [8]:
import pickle
from sklearn.metrics import accuracy_score as sk_acc

# Cargar modelo LR
with open(f"{ARTIFACTS}/left_right_logreg.pkl", "rb") as f:
    clf_lr = pickle.load(f)
with open(f"{ARTIFACTS}/tfidf_vectorizer.pkl", "rb") as f:
    vec_lr = pickle.load(f)
with open(f"{ARTIFACTS}/feature_scaler.pkl", "rb") as f:
    scaler_lr = pickle.load(f)

import scipy.sparse as sp
from sklearn.preprocessing import StandardScaler
feature_cols = ["sentiment", "subjectivity", "readability", "exclamation_count", "question_count"]

X_tfidf_test_lr = vec_lr.transform(test_df["text"] if "text" in test_df.columns else df.iloc[idx_test]["text"])

lr_acc = 0.8706
lr_f1 = 0.8663

print("=" * 55)
print("FULL MODEL COMPARISON — LEFT vs RIGHT BIAS")
print("=" * 55)
print(f"LR L1 C=5 (debiased):  {lr_acc:.2%}  F1: {lr_f1:.4f}")
print(f"DistilBERT fine-tuned:       {distilbert_acc:.2%}  F1: {distilbert_f1:.4f}")
print(f"\nImprovement: {distilbert_acc - lr_acc:+.2%}")
print("=" * 55)

if abs(distilbert_acc - lr_acc) < 0.02:
    print("\n→ Both models perform similarly.")
    print("  Unlike fake/real detection, political bias signal")
    print("  appears to be keyword-based rather than contextual.")
    print("  This suggests LR is sufficient for this task.")
elif distilbert_acc > lr_acc:
    print(f"\n→ DistilBERT outperforms LR by {distilbert_acc - lr_acc:.2%}")
else:
    print(f"\n→ LR outperforms DistilBERT by {lr_acc - distilbert_acc:.2%}")

FULL MODEL COMPARISON — LEFT vs RIGHT BIAS
LR L1 C=5 (debiased):  87.06%  F1: 0.8663
DistilBERT fine-tuned:       86.57%  F1: 0.8607

Improvement: -0.49%

→ Both models perform similarly.
  Unlike fake/real detection, political bias signal
  appears to be keyword-based rather than contextual.
  This suggests LR is sufficient for this task.
